In [1]:
from transformers import AutoProcessor, AutoModelForMultimodalLM

/home/drd92/.conda/envs/vlms/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-2B-Instruct")
model = AutoModelForMultimodalLM.from_pretrained("Qwen/Qwen3-VL-2B-Instruct", device_map="auto")

Loading weights: 100%|██████████| 625/625 [00:03<00:00, 202.46it/s]


In [3]:
model

Qwen3VLForConditionalGeneration(
  (model): Qwen3VLModel(
    (visual): Qwen3VLVisionModel(
      (patch_embed): Qwen3VLVisionPatchEmbed(
        (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 1024)
      (rotary_pos_emb): Qwen3VLVisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-23): 24 x Qwen3VLVisionBlock(
          (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
          (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
          (attn): Qwen3VLVisionAttention(
            (qkv): Linear(in_features=1024, out_features=3072, bias=True)
            (proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (mlp): Qwen3VLVisionMLP(
            (linear_fc1): Linear(in_features=1024, out_features=4096, bias=True)
            (linear_fc2): Linear(in_features=4096, out_features=1024, bias=True)
            (act_fn): GELUTanh()
         

In [4]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"},
            {"type": "text", "text": "What animal is on the candy?"}
        ]
    },
]
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)
inputs

{'input_ids': tensor([[151644,    872,    198,  ..., 151644,  77091,    198]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1]], device='cuda:0'), 'mm_token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0]], device='cuda:0'), 'pixel_values': tensor([[ 0.4588,  0.5059,  0.5294,  ...,  0.5373,  0.5608,  0.5686],
        [ 0.5373,  0.5137,  0.5137,  ...,  0.5216,  0.5451,  0.5216],
        [ 0.5451,  0.5451,  0.5451,  ...,  0.5922,  0.5686,  0.5608],
        ...,
        [-0.3020, -0.2549, -0.2706,  ..., -0.1765, -0.1686, -0.2000],
        [-0.1922, -0.1451, -0.1216,  ..., -0.0275, -0.1059, -0.0980],
        [-0.0510, -0.0510, -0.0588,  ..., -0.1059, -0.1059, -0.1216]],
       device='cuda:0'), 'image_grid_thw': tensor([[  1, 188, 252]], device='cuda:0')}

In [5]:
visual_param_budget = sum(p.numel() for p in model.model.visual.parameters())
text_param_budget = sum(p.numel() for p in model.model.language_model.parameters())
total = sum(p.numel() for p in model.model.parameters())

print(f'  Vision: {visual_param_budget:13,}')
print(f'Language: {text_param_budget:13,}')
print(f'   Model: {total:13,} == {visual_param_budget + text_param_budget:13,}')

  Vision:   406,957,056
Language: 1,720,574,976
   Model: 2,127,532,032 == 2,127,532,032


In [6]:
visual_param_budget / total, text_param_budget / total

(0.19128128266883834, 0.8087187173311616)

In [8]:
for name, p in model.model.named_parameters():
    if p.requires_grad:
        print(f'\ttrainable {name} {p.size()}')

for name, p in model.model.named_parameters():
    if not p.requires_grad:
        print(f'\tfrozen {name} {p.size()}')

	trainable visual.patch_embed.proj.weight torch.Size([1024, 3, 2, 16, 16])
	trainable visual.patch_embed.proj.bias torch.Size([1024])
	trainable visual.pos_embed.weight torch.Size([2304, 1024])
	trainable visual.blocks.0.norm1.weight torch.Size([1024])
	trainable visual.blocks.0.norm1.bias torch.Size([1024])
	trainable visual.blocks.0.norm2.weight torch.Size([1024])
	trainable visual.blocks.0.norm2.bias torch.Size([1024])
	trainable visual.blocks.0.attn.qkv.weight torch.Size([3072, 1024])
	trainable visual.blocks.0.attn.qkv.bias torch.Size([3072])
	trainable visual.blocks.0.attn.proj.weight torch.Size([1024, 1024])
	trainable visual.blocks.0.attn.proj.bias torch.Size([1024])
	trainable visual.blocks.0.mlp.linear_fc1.weight torch.Size([4096, 1024])
	trainable visual.blocks.0.mlp.linear_fc1.bias torch.Size([4096])
	trainable visual.blocks.0.mlp.linear_fc2.weight torch.Size([1024, 4096])
	trainable visual.blocks.0.mlp.linear_fc2.bias torch.Size([1024])
	trainable visual.blocks.1.norm1.wei

In [9]:
model.model.visual.pos_embed.weight

Parameter containing:
tensor([[-0.0693,  0.4062, -0.6797,  ...,  0.4512, -0.2812, -0.5312],
        [-0.0835,  0.0859, -0.8086,  ...,  0.2598, -0.3770, -0.5664],
        [-0.1050, -0.0874, -0.7344,  ...,  0.2617, -0.3164, -0.4375],
        ...,
        [-0.0762,  0.2490, -0.5664,  ...,  0.1641, -0.0087, -0.1562],
        [-0.1206,  0.2090, -0.7070,  ...,  0.1807,  0.0240, -0.0918],
        [-0.1230,  0.2461, -0.4922,  ...,  0.2227,  0.0201, -0.1982]],
       device='cuda:0', dtype=torch.bfloat16, requires_grad=True)